# Train All Experiments (5 Models x 3 Seeds)

Models: A (single-task species), B (single-task freshness), C-EW, C-UW, C-DWA (multi-task, one loss-weighting strategy each). Checkpoints and per-run history save to Drive, not the ephemeral Colab clone, and already-checkpointed runs are skipped so an interrupted session can resume by re-running this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'

import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

!unzip -q "$DATASET_ZIP" -d /content/data

In [ ]:
import sys
sys.path.append('/content/repo/04_Src')

import pandas as pd
from train import train_single_task, train_multitask, TrainConfig

train_df = pd.read_csv('/content/repo/02_Manifests/train.csv')
val_df = pd.read_csv('/content/repo/02_Manifests/val.csv')
len(train_df), len(val_df)

In [ ]:
SEEDS = [0, 1, 2]

single_task_experiments = [
    {'name': 'ModelA_species', 'task': 'species'},
    {'name': 'ModelB_freshness', 'task': 'freshness'},
]
multitask_experiments = [
    {'name': 'ModelC_EW', 'strategy': 'EW'},
    {'name': 'ModelC_UW', 'strategy': 'UW'},
    {'name': 'ModelC_DWA', 'strategy': 'DWA'},
]

In [ ]:
import json

cfg = TrainConfig()
summary_rows = []

for exp in single_task_experiments:
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
        if os.path.exists(ckpt_path):
            continue
        result = train_single_task(
            exp['task'], train_df, val_df, DATASET_ROOT, ckpt_path, seed=seed, cfg=cfg
        )
        summary_rows.append({
            'run_name': run_name, 'model': exp['name'], 'seed': seed,
            'best_val_loss': result['best_val_loss'],
            'training_time_sec': result['training_time_sec'],
            'epochs_trained': len(result['history']),
        })
        with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
            json.dump(result['history'], f)

for exp in multitask_experiments:
    for seed in SEEDS:
        run_name = f"{exp['name']}_seed{seed}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
        if os.path.exists(ckpt_path):
            continue
        result = train_multitask(
            exp['strategy'], train_df, val_df, DATASET_ROOT, ckpt_path, seed=seed, cfg=cfg
        )
        summary_rows.append({
            'run_name': run_name, 'model': exp['name'], 'seed': seed,
            'best_val_loss': result['best_val_loss'],
            'training_time_sec': result['training_time_sec'],
            'epochs_trained': len(result['history']),
        })
        with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
            json.dump(result['history'], f)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'training_summary.csv'), index=False)
summary_df

In [ ]:
!cp "$RESULTS_DIR/training_summary.csv" /content/repo/06_Results/metrics/training_summary.csv